# 07 — Classification Analysis

Comprehensive analysis of the classified news articles dataset.  
Explores topic distributions over time, prediction confidence, class confusion patterns,
per-domain coverage, and provides an interactive class explorer for manual quality checks.

**Input:** `local_data/all_articles_classified.csv`  
**Output:** Analysis plots saved to `local_data/analysis_plots/`

**Prerequisites:** Completed classification run (notebook 06).

In [ ]:
import sys
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import matplotlib.ticker as mticker
import seaborn as sns
from scipy.stats import entropy

# Pipeline utilities
PIPELINE_DIR = str(Path(".").resolve())
if PIPELINE_DIR not in sys.path:
    sys.path.insert(0, PIPELINE_DIR)

import importlib
import pipeline_utils as pu
importlib.reload(pu)

# ════════════════════════════════════════
# CONFIGURATION
# ════════════════════════════════════════
SAVE_PLOTS = True  # Set to False to disable saving plots to disk
# ════════════════════════════════════════

# Plot settings
plt.rcParams.update({
    "figure.dpi": 120,
    "savefig.dpi": 150,
    "savefig.bbox": "tight",
    "font.size": 11,
    "axes.titlesize": 13,
    "axes.labelsize": 11,
})
sns.set_style("whitegrid")

# Consistent color palette for 13 classes
ALL_LABELS = pu.DEFAULT_CANDIDATE_LABELS
_tab20 = plt.cm.tab20.colors
CLASS_COLORS = {label: _tab20[i] for i, label in enumerate(ALL_LABELS)}
CLASS_COLORS["Andere"] = (0.6, 0.6, 0.6)  # grey so it doesn't dominate

# Score column <-> label mapping
CSV_PATH = Path("local_data/all_articles_classified.csv")
SCORE_COLUMNS = [
    "score_Klima_Energie", "score_Zuwanderung", "score_Renten",
    "score_Soziales_Gefaelle", "score_AfD_Rechte", "score_Arbeitslosigkeit",
    "score_Wirtschaftslage", "score_Politikverdruss",
    "score_Gesundheitswesen_Pflege", "score_Kosten_Loehne_Preise",
    "score_Ukraine_Krieg_Russland", "score_Bundeswehr_Verteidigung",
    "score_Andere",
]
SCORE_COL_TO_LABEL = dict(zip(SCORE_COLUMNS, ALL_LABELS))
LABEL_TO_SCORE_COL = dict(zip(ALL_LABELS, SCORE_COLUMNS))

# Output directory
PLOT_DIR = Path("local_data/analysis_plots")
PLOT_DIR.mkdir(parents=True, exist_ok=True)


def savefig(fig, name):
    """Save figure to PLOT_DIR (if enabled) and display."""
    if SAVE_PLOTS:
        fig.savefig(PLOT_DIR / f"{name}.png")
    plt.show()


print(f"Plot output:   {PLOT_DIR.resolve()}")
print(f"Save plots:    {SAVE_PLOTS}")
print(f"Labels:        {len(ALL_LABELS)}")
print(f"CSV:           {CSV_PATH.resolve()}")
print("Setup complete.")

In [ ]:
# Load & prepare data
df = pd.read_csv(CSV_PATH, low_memory=False)

# Parse datetime (mixed ISO 8601: some have milliseconds)
df["datetime"] = pd.to_datetime(df["date_time"], format="ISO8601", utc=True)
df["date"] = df["datetime"].dt.normalize()  # midnight-aligned for grouping

# Derived columns
df["has_label"] = df["label"].notna() & (df["label"] != "")

# Score matrix as numpy for fast vectorised ops
score_matrix = df[SCORE_COLUMNS].values  # (N, 13)

# Second-highest score and its label
sorted_indices = np.argsort(-score_matrix, axis=1)
df["top2_label"] = [ALL_LABELS[sorted_indices[i, 1]] for i in range(len(df))]
df["top2_score"] = score_matrix[np.arange(len(df)), sorted_indices[:, 1]]
df["top1_top2_margin"] = df["prediction_score"] - df["top2_score"]

# Entropy across all 13 scores (higher = more uncertain)
df["score_entropy"] = entropy(score_matrix, axis=1)

print(f"Articles:       {len(df):,}")
print(f"Date range:     {df['date'].min().date()} to {df['date'].max().date()}")
print(f"Unique dates:   {df['date'].nunique()}")
print(f"Unique domains: {df['domain'].nunique()}")
print(f"Labeled subset: {df['has_label'].sum():,}  (train + test)")
print(f"Unlabeled (raw):{(~df['has_label']).sum():,}")
print(f"\nPrediction score  mean={df['prediction_score'].mean():.3f}  "
      f"median={df['prediction_score'].median():.3f}  "
      f"min={df['prediction_score'].min():.3f}  "
      f"max={df['prediction_score'].max():.3f}")

---
## 1. Coverage Over Time (All Classes)

In [ ]:
# Group by date and prediction, count articles
daily_counts = df.groupby(["date", "prediction"]).size().unstack(fill_value=0)

# Ensure all 13 labels are columns
for label in ALL_LABELS:
    if label not in daily_counts.columns:
        daily_counts[label] = 0
daily_counts = daily_counts[ALL_LABELS]

# Convert to percentage share per day
daily_pct = daily_counts.div(daily_counts.sum(axis=1), axis=0) * 100

fig, ax = plt.subplots(figsize=(18, 8))
for label in ALL_LABELS:
    ax.plot(
        daily_pct.index, daily_pct[label],
        label=label, color=CLASS_COLORS[label],
        linewidth=1.5 if label != "Andere" else 2.5,
        alpha=0.9 if label != "Andere" else 0.6,
        linestyle="-" if label != "Andere" else "--",
    )

ax.xaxis.set_major_locator(mdates.WeekdayLocator(byweekday=mdates.MO))
ax.xaxis.set_major_formatter(mdates.DateFormatter("%d.%m"))
ax.xaxis.set_minor_locator(mdates.DayLocator())
ax.set_xlabel("Date")
ax.set_ylabel("Share of Coverage (%)")
ax.set_title("Topic Distribution per Day (All 13 Categories)", fontweight="bold")
ax.legend(bbox_to_anchor=(1.02, 1), loc="upper left", fontsize=9)
ax.set_xlim(daily_pct.index.min(), daily_pct.index.max())
ax.grid(True, alpha=0.3)
plt.tight_layout()
savefig(fig, "01_coverage_over_time_all_classes")

---
## 2. Coverage Over Time (Without "Andere", Normalized to 100%)

In [ ]:
labels_no_andere = [l for l in ALL_LABELS if l != "Andere"]
daily_no_andere = daily_counts[labels_no_andere]
daily_no_andere_pct = daily_no_andere.div(daily_no_andere.sum(axis=1), axis=0) * 100

fig, ax = plt.subplots(figsize=(18, 8))
for label in labels_no_andere:
    ax.plot(
        daily_no_andere_pct.index, daily_no_andere_pct[label],
        label=label, color=CLASS_COLORS[label],
        linewidth=1.5, alpha=0.9,
    )

ax.xaxis.set_major_locator(mdates.WeekdayLocator(byweekday=mdates.MO))
ax.xaxis.set_major_formatter(mdates.DateFormatter("%d.%m"))
ax.xaxis.set_minor_locator(mdates.DayLocator())
ax.set_xlabel("Date")
ax.set_ylabel("Normalized Share (%)")
ax.set_title('Topic Distribution per Day (Excl. "Andere", Normalized to 100%)', fontweight="bold")
ax.legend(bbox_to_anchor=(1.02, 1), loc="upper left", fontsize=9)
ax.set_xlim(daily_no_andere_pct.index.min(), daily_no_andere_pct.index.max())
ax.grid(True, alpha=0.3)
plt.tight_layout()
savefig(fig, "02_coverage_over_time_no_andere")

---
## 2b. News Coverage vs. MIP (Most Important Problem) per Topic

Comparison of daily news coverage share (normalized, excl. "Andere") with survey-based
MIP data. Each subplot shows one topic with both lines overlaid.

In [ ]:
# Load MIP (Most Important Problem) survey data
REPO_ROOT = Path(PIPELINE_DIR).parents[1]
MIP_PATH = REPO_ROOT / "data" / "most_important_problem" / "mip_wide_dataframe.csv"
mip_raw = pd.read_csv(MIP_PATH, na_values="NA")
mip_raw["date"] = pd.to_datetime(mip_raw["date"])

# MIP columns match labels_no_andere (12 topics, no "Andere")
# Normalize MIP per survey date to sum to 100% (comparable with news coverage)
mip_labels = [l for l in labels_no_andere if l in mip_raw.columns]
mip = mip_raw[["date"] + mip_labels].dropna(subset=mip_labels, how="all").copy()
mip_totals = mip[mip_labels].sum(axis=1)
mip_pct = mip[mip_labels].div(mip_totals, axis=0) * 100
mip_pct["date"] = mip["date"].values

print(f"MIP CSV:          {MIP_PATH}")
print(f"MIP survey dates: {len(mip_pct)}")
print(f"Date range:       {mip_pct['date'].min().date()} to {mip_pct['date'].max().date()}")
print(f"Topics matched:   {len(mip_labels)}")

# Ensure both date indices are tz-naive for comparison
news_index = daily_no_andere_pct.index.tz_localize(None) if daily_no_andere_pct.index.tz else daily_no_andere_pct.index
news_values = daily_no_andere_pct.set_index(news_index)
mip_dates = mip_pct["date"].dt.tz_localize(None) if mip_pct["date"].dt.tz else mip_pct["date"]

# Small multiples: one subplot per topic, dual lines (News Coverage vs MIP)
n_topics = len(mip_labels)
ncols = 4
nrows = (n_topics + ncols - 1) // ncols

fig, axes = plt.subplots(nrows, ncols, figsize=(22, 4 * nrows), sharex=True)
axes_flat = axes.flatten()

# Shared x-axis range covering both datasets
x_min = min(news_index.min(), mip_dates.min())
x_max = max(news_index.max(), mip_dates.max())

for i, label in enumerate(mip_labels):
    ax = axes_flat[i]

    # News coverage line (daily)
    ax.plot(
        news_index, news_values[label],
        color=CLASS_COLORS[label], linewidth=1, alpha=0.6,
        label="News Coverage",
    )

    # MIP survey line (sparse data points connected)
    ax.plot(
        mip_dates, mip_pct[label],
        color="black", linewidth=2, marker="o", markersize=5,
        label="MIP Survey", zorder=5,
    )

    ax.set_title(label, fontsize=11, fontweight="bold")
    ax.xaxis.set_major_formatter(mdates.DateFormatter("%d.%m"))
    ax.tick_params(axis="x", rotation=45, labelsize=7)
    ax.set_xlim(x_min, x_max)
    ax.grid(alpha=0.3)
    ax.set_ylabel("Share (%)", fontsize=8)

    if i == 0:
        ax.legend(fontsize=7, loc="upper left")

# Hide unused subplots
for j in range(n_topics, len(axes_flat)):
    axes_flat[j].set_visible(False)

fig.suptitle(
    "News Coverage vs. MIP Survey Data per Topic (Normalized to 100%, Excl. \"Andere\")",
    fontsize=14, fontweight="bold",
)
plt.tight_layout()
savefig(fig, "02b_news_vs_mip_per_topic")

---
## 3. Overall Distribution (Pie Chart)

In [ ]:
pred_counts = df["prediction"].value_counts().reindex(ALL_LABELS).fillna(0)

fig, ax = plt.subplots(figsize=(10, 10))
wedges, texts, autotexts = ax.pie(
    pred_counts.values,
    labels=None,
    autopct=lambda pct: f"{pct:.1f}%" if pct > 3 else "",
    colors=[CLASS_COLORS[l] for l in pred_counts.index],
    startangle=90,
    pctdistance=0.8,
)
for autotext in autotexts:
    autotext.set_fontsize(9)

legend_labels = [f"{label}  ({int(count):,})" for label, count in pred_counts.items()]
ax.legend(wedges, legend_labels, loc="center left",
          bbox_to_anchor=(1.0, 0.5), fontsize=9)
ax.set_title(f"Overall Category Distribution (n={len(df):,})", fontweight="bold")
plt.tight_layout()
savefig(fig, "03_pie_chart_overall")

---
## 4. Mean Confidence Per Class

In [ ]:
conf_by_class = df.groupby("prediction")["prediction_score"].agg(["mean", "std", "count"])
conf_by_class = conf_by_class.reindex(ALL_LABELS)

fig, ax = plt.subplots(figsize=(14, 6))
x = np.arange(len(ALL_LABELS))
bars = ax.bar(
    x, conf_by_class["mean"],
    yerr=conf_by_class["std"], capsize=3,
    color=[CLASS_COLORS[l] for l in ALL_LABELS], alpha=0.85,
    edgecolor="white", linewidth=0.5,
)

for i, (mean_val, std_val, count) in enumerate(
    zip(conf_by_class["mean"], conf_by_class["std"], conf_by_class["count"])
):
    ax.text(i, mean_val + std_val + 0.01,
            f"{mean_val:.3f}\n(n={int(count):,})",
            ha="center", va="bottom", fontsize=8)

ax.set_xticks(x)
ax.set_xticklabels(ALL_LABELS, rotation=45, ha="right", fontsize=9)
ax.set_ylabel("Mean Confidence (prediction_score)")
ax.set_title("Average Confidence per Category", fontweight="bold")
ax.set_ylim(0, 1.15)
ax.grid(axis="y", alpha=0.3)
plt.tight_layout()
savefig(fig, "04_mean_confidence_per_class")

---
## 5. Label Overlap / Confusion Analysis (Top-2 Scores)

In [ ]:
# 5a: Top-2 Confusion Heatmap
# Rows = predicted class, Columns = 2nd-highest class (row-normalized %)

top2_confusion = pd.crosstab(
    df["prediction"], df["top2_label"], normalize="index"
) * 100
top2_confusion = top2_confusion.reindex(index=ALL_LABELS, columns=ALL_LABELS, fill_value=0)

fig, ax = plt.subplots(figsize=(14, 12))
mask = np.eye(len(ALL_LABELS), dtype=bool)
sns.heatmap(
    top2_confusion, annot=True, fmt=".1f", cmap="YlOrRd",
    mask=mask, ax=ax, cbar_kws={"label": "Share (%)", "shrink": 0.8},
    linewidths=0.5, linecolor="white",
    xticklabels=[l[:20] for l in ALL_LABELS],
    yticklabels=[l[:20] for l in ALL_LABELS],
)
ax.set_xlabel("Second-Highest Category (Top-2)")
ax.set_ylabel("Predicted Category (Top-1)")
ax.set_title(
    "Confusion Matrix: Top-2 Scores\n"
    "(Row = predicted, Column = second-highest confidence)",
    fontweight="bold",
)
ax.tick_params(axis="x", rotation=45)
ax.tick_params(axis="y", rotation=0)
plt.tight_layout()
savefig(fig, "05a_top2_confusion_heatmap")

In [ ]:
# 5b: Low-Margin Analysis

fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# Left: histogram of margins
axes[0].hist(df["top1_top2_margin"], bins=50, color="#4C72B0", alpha=0.85, edgecolor="white")
axes[0].axvline(x=0.10, color="red", linestyle="--", label="Threshold: 10%")
axes[0].set_xlabel("Top-1 minus Top-2 Score Difference")
axes[0].set_ylabel("Number of Articles")
axes[0].set_title("Distribution of Top-1/Top-2 Confidence Margin")
axes[0].legend()
axes[0].grid(alpha=0.3)

# Right: top-15 confusion pairs among low-margin articles
low_margin = df[df["top1_top2_margin"] < 0.10]
pair_counts = (
    low_margin.groupby(["prediction", "top2_label"]).size()
    .reset_index(name="count")
    .sort_values("count", ascending=False)
    .head(15)
)
pair_labels = [
    f"{row['prediction'][:15]} vs {row['top2_label'][:15]}"
    for _, row in pair_counts.iterrows()
]
axes[1].barh(range(len(pair_counts)), pair_counts["count"], color="#DD8452", alpha=0.85)
axes[1].set_yticks(range(len(pair_counts)))
axes[1].set_yticklabels(pair_labels, fontsize=8)
axes[1].set_xlabel("Number of Articles")
axes[1].set_title(f"Most Common Confusion Pairs (Margin < 10%, n={len(low_margin):,})")
axes[1].grid(axis="x", alpha=0.3)

fig.suptitle(
    "Analysis: Uncertain Predictions (Low Top-1/Top-2 Margin)",
    fontweight="bold", fontsize=13,
)
plt.tight_layout()
savefig(fig, "05b_low_margin_analysis")

print(f"\nArticles with Top-1/Top-2 margin < 10%: {len(low_margin):,} "
      f"({len(low_margin)/len(df)*100:.1f}%)")
print(f"\nTop 15 confusion pairs:")
for _, row in pair_counts.iterrows():
    print(f"  {row['prediction']:<30} vs {row['top2_label']:<30} n={row['count']}")

In [ ]:
# 5c: Symmetric Confusion Heatmap
# Bidirectional: how often do classes A and B appear as each other's top-2?

pair_counts_all = df.groupby(["prediction", "top2_label"]).size().reset_index(name="count")
symmetric = pd.DataFrame(0.0, index=ALL_LABELS, columns=ALL_LABELS)

for _, row in pair_counts_all.iterrows():
    a, b = row["prediction"], row["top2_label"]
    symmetric.loc[a, b] += row["count"]
    symmetric.loc[b, a] += row["count"]

symmetric = symmetric / symmetric.values.sum() * 100

fig, ax = plt.subplots(figsize=(13, 11))
mask = np.tril(np.ones_like(symmetric, dtype=bool))  # show upper triangle only
sns.heatmap(
    symmetric, annot=True, fmt=".2f", cmap="YlOrRd",
    mask=mask, ax=ax, cbar_kws={"label": "Share (%)", "shrink": 0.8},
    linewidths=0.5, linecolor="white",
    xticklabels=[l[:18] for l in ALL_LABELS],
    yticklabels=[l[:18] for l in ALL_LABELS],
)
ax.set_title("Symmetric Confusion Frequency (Upper Triangle)", fontweight="bold")
ax.tick_params(axis="x", rotation=45)
plt.tight_layout()
savefig(fig, "05c_symmetric_confusion_heatmap")

---
## 6. Per-Class Quality Statistics & Visualizations

In [ ]:
# 6a: Confidence Boxplots

# Sort classes by median confidence
order = df.groupby("prediction")["prediction_score"].median().sort_values().index.tolist()

fig, ax = plt.subplots(figsize=(16, 7))
box_data = [df[df["prediction"] == label]["prediction_score"].values for label in order]
bp = ax.boxplot(
    box_data, vert=True, patch_artist=True,
    labels=[l[:18] for l in order],
    medianprops=dict(color="black", linewidth=1.5),
)
for patch, label in zip(bp["boxes"], order):
    patch.set_facecolor(CLASS_COLORS[label])
    patch.set_alpha(0.75)

ax.set_ylabel("Confidence (prediction_score)")
ax.set_title("Confidence Distribution per Category", fontweight="bold")
ax.tick_params(axis="x", rotation=45)
ax.grid(axis="y", alpha=0.3)
plt.tight_layout()
savefig(fig, "06a_confidence_boxplots")

In [ ]:
# 6b: Confidence Histograms (Small Multiples)

fig, axes = plt.subplots(4, 4, figsize=(18, 14))
axes_flat = axes.flatten()

for i, label in enumerate(ALL_LABELS):
    ax = axes_flat[i]
    subset = df[df["prediction"] == label]["prediction_score"]
    ax.hist(subset, bins=30, color=CLASS_COLORS[label], alpha=0.85, edgecolor="white")
    ax.set_title(f"{label} (n={len(subset):,})", fontsize=10)
    ax.set_xlim(0, 1)
    ax.axvline(
        x=subset.median(), color="black", linestyle="--", linewidth=1,
        label=f"Median: {subset.median():.3f}",
    )
    ax.legend(fontsize=7, loc="upper left")
    ax.grid(alpha=0.3)
    if i >= 12:
        ax.set_xlabel("Confidence")

# Hide empty subplots
for j in range(len(ALL_LABELS), len(axes_flat)):
    axes_flat[j].set_visible(False)

fig.suptitle("Confidence Distributions per Category", fontsize=14, fontweight="bold")
plt.tight_layout()
savefig(fig, "06b_confidence_histograms")

In [ ]:
# 6c: Articles Per Class Over Time (Small Multiples)

fig, axes = plt.subplots(4, 4, figsize=(20, 14), sharex=True)
axes_flat = axes.flatten()

for i, label in enumerate(ALL_LABELS):
    ax = axes_flat[i]
    daily_label = df[df["prediction"] == label].groupby("date").size()
    daily_label = daily_label.reindex(daily_counts.index, fill_value=0)
    ax.fill_between(daily_label.index, daily_label.values,
                    color=CLASS_COLORS[label], alpha=0.5)
    ax.plot(daily_label.index, daily_label.values,
            color=CLASS_COLORS[label], linewidth=1)
    ax.set_title(f"{label}", fontsize=10)
    ax.xaxis.set_major_formatter(mdates.DateFormatter("%d.%m"))
    ax.tick_params(axis="x", rotation=45, labelsize=7)
    ax.grid(alpha=0.3)

for j in range(len(ALL_LABELS), len(axes_flat)):
    axes_flat[j].set_visible(False)

fig.suptitle("Articles per Day by Category", fontsize=14, fontweight="bold")
plt.tight_layout()
savefig(fig, "06c_articles_per_class_over_time")

In [ ]:
# 6d: Low-Confidence Articles Table

thresholds = [0.3, 0.4, 0.5, 0.6, 0.7, 0.8]

print("Articles below various confidence thresholds:\n")
rows = []
for label in ALL_LABELS:
    subset = df[df["prediction"] == label]
    n = len(subset)
    row = {"Category": label, "Total": n}
    for t in thresholds:
        below = (subset["prediction_score"] < t).sum()
        row[f"< {t}"] = f"{below} ({below/n*100:.1f}%)" if n > 0 else "0"
    rows.append(row)

low_conf_table = pd.DataFrame(rows)
print(low_conf_table.to_string(index=False))

print(f"\n\n5 articles with lowest confidence overall:")
lowest = df.nsmallest(5, "prediction_score")[
    ["headline", "prediction", "prediction_score", "top2_label", "top2_score", "domain"]
]
print(lowest.to_string(index=False))

---
## 7. Interactive Class Explorer

Change `EXPLORE_CLASS` below and re-run this cell to inspect articles for any class.

In [ ]:
# ════════════════════════════════════════
# CONFIGURE: Select a class to explore
# ════════════════════════════════════════
EXPLORE_CLASS = "Wirtschaftslage"  # <-- change to any of the 13 labels
# ════════════════════════════════════════

assert EXPLORE_CLASS in ALL_LABELS, (
    f"Unknown class: {EXPLORE_CLASS}. Valid: {ALL_LABELS}"
)

subset = df[df["prediction"] == EXPLORE_CLASS].copy()
print(f"Category:          {EXPLORE_CLASS}")
print(f"Articles:          {len(subset):,}")
print(f"Mean confidence:   {subset['prediction_score'].mean():.4f}")
print(f"Median confidence: {subset['prediction_score'].median():.4f}")
print()

# --- Top 10 highest confidence ---
top10 = subset.nlargest(10, "prediction_score")
print("=" * 80)
print(f"  TOP 10: HIGHEST CONFIDENCE ({EXPLORE_CLASS})")
print("=" * 80)

for rank, (_, row) in enumerate(top10.iterrows(), 1):
    print(f"\n--- #{rank} | Confidence: {row['prediction_score']:.4f} "
          f"| Domain: {row['domain']} ---")
    print(f"Headline: {row['headline']}")
    print(f"Date: {row['date_time']}")
    text = str(row["text"])
    print(f"Text ({len(text)} chars):\n{text[:1000]}{'...' if len(text) > 1000 else ''}")

print("\n\n")

# --- Top 10 lowest confidence ---
bottom10 = subset.nsmallest(10, "prediction_score")
print("=" * 80)
print(f"  TOP 10: LOWEST CONFIDENCE ({EXPLORE_CLASS})")
print("=" * 80)

for rank, (_, row) in enumerate(bottom10.iterrows(), 1):
    print(f"\n--- #{rank} | Confidence: {row['prediction_score']:.4f} "
          f"| Domain: {row['domain']} ---")
    print(f"Headline: {row['headline']}")
    print(f"Date: {row['date_time']}")
    print(f"Top-2: {row['top2_label']} ({row['top2_score']:.4f})")
    text = str(row["text"])
    print(f"Text ({len(text)} chars):\n{text[:1000]}{'...' if len(text) > 1000 else ''}")

---
## 8. Additional Analyses

In [ ]:
# 8a: Coverage by Domain (Top 15 Domains x 13 Classes Heatmap)

top_domains = df["domain"].value_counts().head(15).index.tolist()
domain_subset = df[df["domain"].isin(top_domains)]

domain_topic = pd.crosstab(
    domain_subset["domain"], domain_subset["prediction"], normalize="index"
) * 100
domain_topic = domain_topic.reindex(columns=ALL_LABELS, fill_value=0)
domain_topic = domain_topic.loc[top_domains]

fig, ax = plt.subplots(figsize=(16, 10))
sns.heatmap(
    domain_topic, annot=True, fmt=".1f", cmap="Blues",
    ax=ax, cbar_kws={"label": "Share (%)", "shrink": 0.8},
    linewidths=0.5, linecolor="white",
    xticklabels=[l[:18] for l in ALL_LABELS],
)
ax.set_xlabel("Category")
ax.set_ylabel("Domain")
ax.set_title(
    "Topic Coverage by Domain (Top 15 Domains, Row-Normalized)",
    fontweight="bold",
)
ax.tick_params(axis="x", rotation=45)
plt.tight_layout()
savefig(fig, "08a_coverage_by_domain")

In [ ]:
# 8b: Confidence vs Text Length

fig, axes = plt.subplots(1, 2, figsize=(16, 6))

sample = df.sample(min(5000, len(df)), random_state=42)

# Left: by character length
axes[0].scatter(
    sample["text_length_char"], sample["prediction_score"],
    alpha=0.15, s=5, c="#4C72B0",
)
axes[0].set_xlabel("Text Length (Characters)")
axes[0].set_ylabel("Confidence")
axes[0].set_title("Confidence vs. Text Length (Characters)")
axes[0].grid(alpha=0.3)

# Right: by token length (truncation at 2048)
axes[1].scatter(
    sample["text_length_token"], sample["prediction_score"],
    alpha=0.15, s=5, c="#DD8452",
)
axes[1].axvline(x=2048, color="red", linestyle="--", alpha=0.7, label="Truncation (2048)")
axes[1].set_xlabel("Text Length (Tokens)")
axes[1].set_ylabel("Confidence")
axes[1].set_title("Confidence vs. Text Length (Tokens)")
axes[1].legend()
axes[1].grid(alpha=0.3)

corr_char = df["text_length_char"].corr(df["prediction_score"])
corr_token = df["text_length_token"].corr(df["prediction_score"])
fig.suptitle(
    f"Text Length vs. Confidence  (r_char={corr_char:.3f}, r_token={corr_token:.3f})",
    fontsize=13, fontweight="bold",
)
plt.tight_layout()
savefig(fig, "08b_confidence_vs_text_length")

In [ ]:
# 8c: Daily Article Volume

daily_total = df.groupby("date").size()

fig, ax = plt.subplots(figsize=(16, 5))

# Color weekends differently
colors = [
    "#E57373" if d.weekday() >= 5 else "#4C72B0"
    for d in daily_total.index
]

ax.bar(daily_total.index, daily_total.values, color=colors, alpha=0.85, width=0.8)
ax.axhline(
    y=daily_total.mean(), color="black", linestyle="--", alpha=0.5,
    label=f"Mean: {daily_total.mean():.0f}",
)
ax.xaxis.set_major_locator(mdates.WeekdayLocator(byweekday=mdates.MO))
ax.xaxis.set_major_formatter(mdates.DateFormatter("%d.%m"))
ax.set_xlabel("Date")
ax.set_ylabel("Number of Articles")
ax.set_title(
    "Daily Article Volume (Blue = Weekday, Red = Weekend)",
    fontweight="bold",
)
ax.legend()
ax.grid(axis="y", alpha=0.3)
plt.tight_layout()
savefig(fig, "08c_daily_article_volume")

In [ ]:
# 8d: Stacked Area Chart (without "Andere")

# Stack order: largest mean share at bottom
mean_share = daily_no_andere_pct.mean().sort_values(ascending=False)
stack_order = mean_share.index.tolist()

fig, ax = plt.subplots(figsize=(18, 8))
ax.stackplot(
    daily_no_andere_pct.index,
    [daily_no_andere_pct[label] for label in stack_order],
    labels=stack_order,
    colors=[CLASS_COLORS[l] for l in stack_order],
    alpha=0.8,
)

ax.xaxis.set_major_locator(mdates.WeekdayLocator(byweekday=mdates.MO))
ax.xaxis.set_major_formatter(mdates.DateFormatter("%d.%m"))
ax.set_xlabel("Date")
ax.set_ylabel("Share (%)")
ax.set_title(
    'Topic Composition Over Time (Stacked, Excl. "Andere")',
    fontweight="bold",
)
ax.legend(bbox_to_anchor=(1.02, 1), loc="upper left", fontsize=9)
ax.set_xlim(daily_no_andere_pct.index.min(), daily_no_andere_pct.index.max())
ax.set_ylim(0, 100)
ax.grid(alpha=0.3)
plt.tight_layout()
savefig(fig, "08d_stacked_area_chart")

In [ ]:
# 8e: Entropy Analysis

fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# Left: Entropy distribution
axes[0].hist(df["score_entropy"], bins=50, color="#8172B3", alpha=0.85, edgecolor="white")
axes[0].set_xlabel("Entropy (nat)")
axes[0].set_ylabel("Number of Articles")
axes[0].set_title("Score Entropy Distribution")
axes[0].grid(alpha=0.3)

max_entropy = np.log(13)
axes[0].axvline(
    x=max_entropy, color="red", linestyle="--",
    label=f"Max Entropy: {max_entropy:.2f}",
)
axes[0].legend()

# Right: Mean entropy per predicted class
mean_ent = df.groupby("prediction")["score_entropy"].mean().reindex(ALL_LABELS)
x = np.arange(len(ALL_LABELS))
axes[1].bar(x, mean_ent.values, color=[CLASS_COLORS[l] for l in ALL_LABELS], alpha=0.85)
axes[1].set_xticks(x)
axes[1].set_xticklabels(ALL_LABELS, rotation=45, ha="right", fontsize=8)
axes[1].set_ylabel("Mean Entropy")
axes[1].set_title("Mean Entropy per Category")
axes[1].grid(axis="y", alpha=0.3)

fig.suptitle("Entropy Analysis (Model Uncertainty)", fontsize=13, fontweight="bold")
plt.tight_layout()
savefig(fig, "08e_entropy_analysis")

print("\n10 articles with highest entropy (maximum uncertainty):")
high_entropy = df.nlargest(10, "score_entropy")[
    ["headline", "prediction", "prediction_score",
     "top2_label", "top2_score", "score_entropy", "domain"]
]
print(high_entropy.to_string(index=False))

In [ ]:
# 8f: Confidence Threshold Analysis

thresholds = np.arange(0.1, 1.0, 0.05)
overall_retained = [(df["prediction_score"] >= t).mean() * 100 for t in thresholds]

fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# Left: Overall retention curve
axes[0].plot(thresholds, overall_retained, "o-", color="#4C72B0", markersize=4)
axes[0].set_xlabel("Confidence Threshold")
axes[0].set_ylabel("Retained Articles (%)")
axes[0].set_title("Proportion of Articles Retained at Various Thresholds")
axes[0].grid(alpha=0.3)

for t_mark in [0.5, 0.7, 0.9]:
    retained_at_t = (df["prediction_score"] >= t_mark).mean() * 100
    axes[0].annotate(
        f"{retained_at_t:.1f}% at {t_mark}",
        xy=(t_mark, retained_at_t), fontsize=8,
        arrowprops=dict(arrowstyle="->", color="gray"),
        textcoords="offset points", xytext=(15, 10),
    )

# Right: Per-class at threshold=0.5
retained_per_class = {}
for label in ALL_LABELS:
    sub = df[df["prediction"] == label]
    retained_per_class[label] = (sub["prediction_score"] >= 0.5).mean() * 100 if len(sub) > 0 else 0

x = np.arange(len(ALL_LABELS))
axes[1].bar(
    x, [retained_per_class[l] for l in ALL_LABELS],
    color=[CLASS_COLORS[l] for l in ALL_LABELS], alpha=0.85,
)
axes[1].set_xticks(x)
axes[1].set_xticklabels(ALL_LABELS, rotation=45, ha="right", fontsize=8)
axes[1].set_ylabel("Retained Articles (%)")
axes[1].set_title("Retained Articles per Category (Threshold = 0.5)")
axes[1].set_ylim(0, 105)
axes[1].grid(axis="y", alpha=0.3)

fig.suptitle("Confidence Threshold Analysis", fontsize=13, fontweight="bold")
plt.tight_layout()
savefig(fig, "08f_confidence_threshold_analysis")

---
## Summary

All analysis plots have been saved to `local_data/analysis_plots/`.

In [ ]:
if SAVE_PLOTS:
    print(f"Saved plots in {PLOT_DIR.resolve()}:\n")
    for p in sorted(PLOT_DIR.glob("*.png")):
        size_kb = p.stat().st_size / 1024
        print(f"  {p.name:<50s} {size_kb:>7.1f} KB")
else:
    print("Plot saving was disabled (SAVE_PLOTS = False).")